# **Convolution and correlation filters**
## Image Processing and Computer Vision


## Correlation filter

Correlation is a filtering operation that slides a kernel over the image. For each pixel, it multiplies the kernel values by the corresponding pixel values in the neighborhood and sums the results. The sum becomes the new value of the pixel at the kernel's center.


In [4]:
import numpy as np
import matplotlib.pyplot as plt

At the image border, part of the kernel extends beyond the image, where no pixel values exist. Without handling this, the operation can only be computed where the kernel fits entirely inside the image, so the output shrinks to (H − k + 1) × (W − k + 1). Padding solves this by adding a frame of ⌊k/2⌋ pixels around the image. The frame is usually filled with zeros, replicated edge values or a mirrored copy of the image. Every pixel then has a full neighbourhood, and the output keeps the same size as the input. The same applies to both convolution and correlation.

In [ ]:
def zero_padding(img: np.array, up_p:int = 0, down_p:int = 0, left_p:int = 0, right_p:int = 0) -> np.array: 
    
    new_row_u = np.zeros((up_p, np.shape(img)[1]))
    img = np.vstack([new_row_u, img])

    new_row_d = np.zeros((down_p, np.shape(img)[1]))
    img = np.vstack([img, new_row_d])

    new_col_l = np.zeros((np.shape(img)[0],left_p))
    img = np.hstack([new_col_l, img])

    new_col_r = np.zeros((np.shape(img)[0],right_p))
    img = np.hstack([img, new_col_r])

    return img

The correlation formula is shows as follows:
$$
g(x, y) = \sum_{s=-a}^{a} \sum_{t=-b}^{b} h(s, t)\, f(x + s, y + t)
$$

For the implementation we assume that:
- Zero padding is used
- The kernel is a matrix with an odd number of rows and columns, so that its center is well defined.

In [73]:
def corr_filter(img:np.array, kernel:np.array) -> np.array:
    k_rows, k_cols = np.shape(kernel)
    img_rows, img_cols = np.shape(img)
    res = np.zeros_like(img)

    # padding size
    up_p, down_p = k_rows//2, k_rows//2
    left_p, right_p = k_cols//2, k_cols//2

    # kernel center of height and kernel center of width  
    kc_r, kc_c = k_rows//2, k_cols//2 # center row, center col

    # PADDING
    
    # 1° method padding: we add zero padding around the image according to the kernel size
    img = zero_padding(img, up_p, down_p, left_p, right_p)

    # 2° method padding: 
    #np.pad(img,((up_p,down_p),(left_p,right_p)))
           
    # CORRELATION

    # 1° method correaltion: from scratch
    for x in range(img_rows):
        for y in range(img_cols):
            conv_res = 0
            for i in range(-(k_rows//2), (k_rows//2) + 1):
                for j in range(-(k_cols//2),(k_cols//2) + 1):
                    conv_res = conv_res + (img[x + kc_r + i, y + kc_c + j] * kernel[kc_r + i, kc_c + j])  
            res[x,y] = conv_res
    

    #2° method correlation: numpy
    """
    for x in range(img_rows):
        for y in range(img_cols):
           res[x, y] = np.sum(img[x:x + k_rows, y:y + k_cols] * kernel)
    """

    return res


    

We test the function using a simple matrix and 

In [74]:
img = np.zeros((5,5))
img[2,2] = 1

kernel = np.array([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
res = corr_filter(img, kernel)


print("Image")
print(img, end = "\n\n")
print("Kernel")
print(kernel, end = "\n\n")
print("Res")
print(res)



Image
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Kernel
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Res
[[0. 0. 0. 0. 0.]
 [0. 9. 8. 7. 0.]
 [0. 6. 5. 4. 0.]
 [0. 3. 2. 1. 0.]
 [0. 0. 0. 0. 0.]]


## Convolution filter

For the convolution of $f$ with $h$ is the correlation of $f$ with the mirrored version of $h$. 
The convolution formula is shows as follows:
$$
g(x, y) = \sum_{s=-a}^{a} \sum_{t=-b}^{b} h(s, t)\, f(x - s, y - t)
$$

For the implementation we assume that:
- Zero padding is used
- The kernel is a matrix with an odd number of rows and columns, so that its center is well defined.

In [ ]:
def conv_filter(img:np.array, kernel:np.array) -> np.array:
    k_rows, k_cols = np.shape(kernel)
    img_rows, img_cols = np.shape(img)
    res = np.zeros_like(img)

    # padding size
    up_p, down_p = k_rows//2, k_rows//2
    left_p, right_p = k_cols//2, k_cols//2

    # kernel center of height and kernel center of width  
    kc_r, kc_c = k_rows//2, k_cols//2 # center row, center col

    # PADDING
    
    # 1° method padding: we add zero padding around the image according to the kernel size
    img = zero_padding(img, up_p, down_p, left_p, right_p)

    # 2° method padding: 
    #np.pad(img,((up_p,down_p),(left_p,right_p)))
           
    # CONVOLUTION
    """
    # 1° method correaltion: from scratch
    for x in range(img_rows):
        for y in range(img_cols):
            conv_res = 0
            for i in range(-(k_rows//2), (k_rows//2) + 1):
                for j in range(-(k_cols//2),(k_cols//2) + 1):
                    conv_res = conv_res + (img[x + kc_r - i, y + kc_c - j] * kernel[kc_r + i, kc_c + j])  
            res[x,y] = conv_res
    """

    #2° method correlation: numpy
    kernel_f = kernel[::-1, ::-1]  # flipped kernel

    for x in range(img_rows):
        for y in range(img_cols):
           res[x, y] = np.sum(img[x:x + k_rows, y:y + k_cols] * kernel_f)
    

    return res


In [84]:
vimg = np.zeros((5,5))
img[2,2] = 1

kernel = np.array([[1,2,3],
                  [4,5,6],
                  [7,8,9]])
res = conv_filter(img, kernel)


print("Image")
print(img, end = "\n\n")
print("Kernel")
print(kernel, end = "\n\n")
print("Res")
print(res)



Image
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

Kernel
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Res
[[0. 0. 0. 0. 0.]
 [0. 1. 2. 3. 0.]
 [0. 4. 5. 6. 0.]
 [0. 7. 8. 9. 0.]
 [0. 0. 0. 0. 0.]]
